# 08 · Supplementary checks (ranking detail, proportional odds, tier mix, descriptives)
**Team DSA · NEXUS DELTA · Build for Bharat 2.0 Round 2**  
Run top-to-bottom (Colab: Runtime → Run all). Notebooks 01→06 share files in `out/`.

In [1]:
import os, json, re, warnings, time
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)
# ---- Colab: upload the 4 organiser files to /content (left sidebar) or mount Drive and change DATA_DIR ----
DATA_DIR = "."
os.makedirs("figs", exist_ok=True); os.makedirs("out", exist_ok=True)
RES = "out/results.json"
def save_res(key, val):
    r = json.load(open(RES)) if os.path.exists(RES) else {}
    r[key] = val
    json.dump(r, open(RES, "w"), indent=1, default=float)
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold"})
C = {"navy": "#12355b", "teal": "#1b998b", "amber": "#e09f3e", "red": "#c8553d", "grey": "#8d99ae"}

## Supplementary checks referenced in the Approach Note
1. Bootstrap ranking detail (probability of ranking first among the funded capabilities).
2. Proportional-odds check: separate binary logits at each salary-band cut-point.
3. Tier-2/3 vs Tier-1 capability prevalence ratios.
4. Descriptive statistics, rank correlations and VIF for JDS / SDS.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
JDS = pd.read_excel(f"{DATA_DIR}/JDS_Skill_Traits.xlsx"); JDS.columns = ["id","big_data","maths_stats","coding","ai_ml","story","high_hike"]
SK = ["big_data","maths_stats","coding","ai_ml","story"]
X, y = JDS[SK].values, JDS.high_hike.values
def ek(X, y, step=0.5, Cc=1.0):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=Cc, max_iter=2000)).fit(X, y); p0 = m.predict_proba(X)[:,1]
    out = []
    for k in range(5):
        Z = X.copy(); Z[:,k] = np.minimum(Z[:,k]+step, 5.0); out.append(100*(m.predict_proba(Z)[:,1]-p0).mean())
    return np.array(out)
rng = np.random.default_rng(SEED)
B = np.array([ek(X[i], y[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(1000))])   # same seed and order as NB06
funded = [SK.index(k) for k in ["maths_stats","coding","ai_ml","story"]]
lead = SK.index("maths_stats")
detail = dict(p_beat_runner_up=float((B[:,lead] > B[:,SK.index("ai_ml")]).mean()),
              p_first_among_funded=float((B[:,funded].argmax(1) == funded.index(lead)).mean()),
              p_first_among_all=float((B.argmax(1) == lead).mean()))
print({k: round(v,3) for k,v in detail.items()}); save_res("decision_detail", detail)

{'p_beat_runner_up': 0.957, 'p_first_among_funded': 0.913, 'p_first_among_all': 0.795}


In [3]:
import statsmodels.api as sm
a = pd.read_csv("out/analytics_clean.csv")
for c in a.columns:
    if a[c].dtype == object and set(a[c].dropna().unique()) <= {"True","False"}: a[c] = a[c].eq("True")
FL = ["big_data","maths_stats","coding","ai_ml","story"]
def design(df):   # S5 controls, primary taxonomy (as NB04)
    X = df[[f"{f}_p" for f in FL]].astype(float).copy(); X.columns = FL
    X["exp_min"] = df.exp_min/5.0
    X = X.join(pd.get_dummies(df.seniority, prefix="sen").drop(columns="sen_mid").astype(float))
    X = X.join(pd.get_dummies(df.loc_tier, prefix="loc").drop(columns="loc_Tier1").astype(float))
    X["exp_span"] = df.exp_span/5.0; X["jobtype_missing"] = df.jobtype_missing.astype(float); X["trunc_skills"] = df.trunc_skills.astype(float)
    return X.join(pd.get_dummies(df.role_family, prefix="role").drop(columns="role_other").astype(float))
P = a[a.is_ds_p]; Xp = sm.add_constant(design(P)); yp = P.salary_ord.values
CUTS = ["3","6","10","15","25"]; po = {}
for j, c in enumerate(CUTS, 1):
    m = sm.Logit((yp >= j).astype(int), Xp).fit(disp=0); ci = m.conf_int()
    po[f">={c}"] = {f: dict(OR=float(np.exp(m.params[f])), lo=float(np.exp(ci.loc[f,0])), hi=float(np.exp(ci.loc[f,1]))) for f in FL}
print(pd.DataFrame({k: {f: f"{v[f]['OR']:.2f} [{v[f]['lo']:.2f}, {v[f]['hi']:.2f}]" for f in FL} for k, v in po.items()}).to_string())
print("AI & ML postings in the lowest band:", int(((P.ai_ml_p == 1) & (P.salary_ord == 0)).sum()))
save_res("po_check", po)

/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


                              >=3                >=6               >=10               >=15               >=25
big_data        2.55 [0.98, 6.64]  1.67 [1.12, 2.48]  1.04 [0.79, 1.37]  0.86 [0.65, 1.15]  1.07 [0.69, 1.66]
maths_stats     1.80 [1.25, 2.59]  1.77 [1.41, 2.22]  1.21 [1.01, 1.44]  1.17 [0.97, 1.41]  0.97 [0.70, 1.33]
coding          2.44 [1.78, 3.35]  1.76 [1.46, 2.13]  1.43 [1.23, 1.67]  1.28 [1.09, 1.50]  1.15 [0.86, 1.52]
ai_ml        22.83 [3.05, 170.84]  1.57 [1.01, 2.42]  1.15 [0.83, 1.59]  1.52 [1.09, 2.12]  1.69 [1.04, 2.76]
story           0.76 [0.60, 0.96]  0.66 [0.55, 0.80]  0.74 [0.63, 0.87]  0.77 [0.63, 0.95]  0.83 [0.56, 1.25]
AI & ML postings in the lowest band: 10


In [4]:
flags = {"Big Data":"big_data_p","Maths & Statistics":"maths_stats_p","Coding":"coding_p","AI & ML":"ai_ml_p","Storytelling (family)":"story_p","Visualisation":"viz_p","MIS / reporting":"mis_p","MIS only":"mis_only_p"}
t1, t23 = P[P.loc_tier=="Tier1"], P[P.loc_tier=="Tier2_3"]
tr = {k: dict(tier1=float(t1[v].astype(float).mean()*100), tier2_3=float(t23[v].astype(float).mean()*100), ratio=float(t23[v].astype(float).mean()/t1[v].astype(float).mean())) for k, v in flags.items()}
print(pd.DataFrame(tr).T.round(2).to_string()); save_res("tier_ratios", tr)

                       tier1  tier2_3  ratio
Big Data                9.83     6.14   0.63
Maths & Statistics     17.82     9.83   0.55
Coding                 30.62    15.97   0.52
AI & ML                15.02     6.88   0.46
Storytelling (family)  16.72    14.74   0.88
Visualisation           5.56     3.69   0.66
MIS / reporting        13.12    11.30   0.86
MIS only               11.16    11.06   0.99


In [5]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
SDS = pd.read_excel(f"{DATA_DIR}/SDS_Personality_Traits.xlsx"); SDS.columns = ["id","neuroticism","extraversion","openness","agreeableness","conscientiousness","success"]
TR = ["neuroticism","extraversion","openness","agreeableness","conscientiousness"]
desc = dict(jds=JDS[SK].agg(["mean","std","min","median","max","skew"]).T.round(3).to_dict("index"),
            jds_at_ceiling={k: float((JDS[k]==5).mean()) for k in SK},
            jds_spearman=JDS[SK].corr(method="spearman").round(3).to_dict(),
            jds_vif=[float(vif(JDS[SK].assign(c=1).values, i)) for i in range(5)],
            sds=SDS[TR].agg(["mean","std","min","median","max"]).T.round(2).to_dict("index"),
            hike_share=float(JDS.high_hike.mean()), success_share=float(SDS.success.mean()))
print(pd.DataFrame(desc["jds"]).T.to_string()); print("VIF", np.round(desc["jds_vif"],2)); save_res("descriptives", desc)

              mean    std  min  median  max   skew
big_data     3.850  0.847  2.3     3.8  5.0 -0.081
maths_stats  4.294  0.844  2.2     4.6  5.0 -0.973
coding       4.268  0.893  2.2     4.6  5.0 -0.890
ai_ml        4.566  0.667  2.2     4.9  5.0 -1.705
story        4.355  0.933  2.3     5.0  5.0 -1.079
VIF [1.04 1.45 1.39 1.22 1.33]
